In [ ]:
!pip install torch==2.3.1 torchvision==0.18.1 --index-url https://download.pytorch.org/whl/cu121 -q
!pip install transformers==4.45.2 -q
!pip install tokenizers==0.20.3 -q
!pip install datasets==2.19.1 -q
!pip install evaluate==0.4.3 -q
!pip install rouge-score==0.1.2 -q
!pip install bert-score==0.3.13 -q
!pip install bitsandbytes==0.43.1 -q
!pip install peft==0.13.2 -q
!pip install accelerate==0.34.2 -q
!pip install torchao==0.5.0 -q
!pip install triton==2.3.1 -q

In [ ]:
token = ''
DRIVE_PATH = '/content/drive/MyDrive/Colab Notebooks/nlp/final'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Data

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorForLanguageModeling, DataCollatorWithPadding

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B-Instruct", token=token)

raw_dataset = load_dataset('json', data_files='/kaggle/input/datasets/nekoza1/data-v4/data-v4.json')
split_dataset = raw_dataset['train'].train_test_split(test_size=29, seed=42)
temp_dataset = split_dataset['train'].train_test_split(test_size=20, seed=42)

train_dataset = temp_dataset['train']
test_dataset = split_dataset['test']
dev_dataset = temp_dataset['test']

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

You are using a model of type llama to instantiate a model of type . This is not supported for all configurations of models and can yield errors.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

#### Pre-processing

In [ ]:
def PROMPT_TEMPLATE (context, question, answer=None):
  context = '\n'.join([
      f'- {context['text']}'
      for context in context.values()
  ])

  prefix = (
      '### Hướng dẫn: Trả lời ngắn gọn, đúng trọng tâm câu hỏi dựa trên ngữ cảnh.\n\n'
      f'### Câu hỏi: {question}\n\n'
      f'### Ngữ cảnh:\n'
  )

  suffix = "\n\n### Trả lời:"

  return {
    'prefix': prefix,
    'suffix': suffix,
    'context': context,
    'answer': answer,
  }

In [ ]:
def preprocess(example):
  return PROMPT_TEMPLATE(example['C'], example['Q'], example['A'])

def tokenize(example):
  max_len = 768

  bos = [tokenizer.bos_token_id]
  eos = [tokenizer.eos_token_id]
  prefix_ids = tokenizer(example['prefix'], add_special_tokens=False)['input_ids']
  suffix_ids = tokenizer(example['suffix'], add_special_tokens=False)['input_ids']
  answer_ids = tokenizer(example['answer'], add_special_tokens=False)['input_ids']
  context_ids = tokenizer(example['context'], add_special_tokens=False)['input_ids']

  fixed_len = (
      len(prefix_ids) +
      len(suffix_ids) +
      len(answer_ids) +
      2
  )

  remaining_len = max_len - fixed_len

  if remaining_len < 0:
    answer_ids = answer_ids[:max_len - len(prefix_ids) - len(suffix_ids)]
    remaining_len = 0

  context_ids = context_ids[:remaining_len]

  prompt_ids = (
    bos +
    prefix_ids +
    context_ids +
    suffix_ids
  )

  prompt_len = len(prompt_ids)

  input_ids = prompt_ids + answer_ids + eos
  labels = [-100] * prompt_len + input_ids[prompt_len:]
  attention_mask = [1] * len(input_ids)

  return {
      'input_ids': input_ids,
      'attention_mask': attention_mask,
      'labels': labels
  }

train_dataset = train_dataset.map(preprocess)
train_dataset = train_dataset.map(tokenize, remove_columns=train_dataset.column_names)

dev_dataset = dev_dataset.map(preprocess)
dev_dataset = dev_dataset.map(tokenize, remove_columns=dev_dataset.column_names)

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

## Model

In [ ]:
import torch
import inspect
import evaluate
import numpy as np
from torch.utils.data import DataLoader
from peft import LoraConfig, get_peft_model, PeftModel, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer, BitsAndBytesConfig


In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

lora_config = LoraConfig(
    r=2,
    lora_alpha=4,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
)

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"        # use only GPU 0
os.environ["TOKENIZERS_PARALLELISM"] = "false"

model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-3B-Instruct", quantization_config=bnb_config, token=token, device_map={'': 0})
tokenizer.add_special_tokens({'pad_token': '<|pad|>'})
model.resize_token_embeddings(len(tokenizer))
model.config.pad_token_id = tokenizer.pad_token_id

model = prepare_model_for_kbit_training(model)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

model.enable_input_require_grads()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

trainable params: 6,078,464 || all params: 3,218,831,360 || trainable%: 0.1888


#### Training Arguments

In [ ]:
training_args = TrainingArguments(
    output_dir=f"/kaggle/working/llm-rag-finetuned-instruct-v1",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    learning_rate=1e-4,
    bf16=True,

    logging_steps=10,
    load_best_model_at_end=True,
    save_strategy="steps",
    save_steps=10,
    eval_strategy="steps",
    eval_steps=10,
    metric_for_best_model="eval_loss",

    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    prediction_loss_only=True,
    warmup_ratio=0.1,
    report_to="none"
)

#### Data Collator

In [ ]:
class MaskedQuestionContextCollator(DataCollatorWithPadding):
  def __call__(self, features):
    labels_list = [f.pop('labels') for f in features]
    input_ids_list = [f['input_ids'] for f in features]

    batch = super().__call__(features)

    padded_labels = []
    max_length = batch['input_ids'].shape[1]

    for idx, label in enumerate(labels_list):
      seq_len = len(input_ids_list[idx])
      pad_len = max_length - seq_len

      padded_label = label + [-100] * pad_len
      padded_labels.append(padded_label)

    batch['labels'] = torch.tensor(padded_labels)
    return batch

#### Trainer

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    tokenizer=tokenizer,
    data_collator=MaskedQuestionContextCollator(tokenizer=tokenizer)
)

trainer.train()


trainer.save_model("/kaggle/working/llm-rag-finetuned-instruct-v1/best_model")
tokenizer.save_pretrained("kaggle/working/llm-rag-finetuned-instruct-v1/tokenizer")

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


Step,Training Loss,Validation Loss


OutOfMemoryError: Caught OutOfMemoryError in replica 0 on device 0.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/parallel_apply.py", line 83, in _worker
    output = module(*input, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1532, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1541, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/peft/peft_model.py", line 1644, in forward
    return self.base_model(
           ^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1532, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^

#### Metrics

In [ ]:
bleu_metric = evaluate.load("bleu")
rouge_metric = evaluate.load("rouge")
bertscore_metric = evaluate.load("bertscore")

def compute_metrics(predicted, label):
  if not predicted.strip():
    return {
        'bleu': 0.0,
        'rougeL': 0.0,
        'bertscore': 0.0
    }

  bleu_score = bleu_metric.compute(predictions=[predicted], references=[[label]], smooth=True)
  rouge_score = rouge_metric.compute(predictions=[predicted], references=[label])
  bertscore_result = bertscore_metric.compute(
        predictions=[predicted],
        references=[label],
        lang="vi"
    )

  bertscore = bertscore_result['f1'][0]

  return {
      'bleu': bleu_score['bleu'],
      'rougeL': rouge_score['rougeL'],
      'bertscore': bertscore
  }

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-3B-Instruct", quantization_config=bnb_config, token=token)
tokenizer = AutoTokenizer.from_pretrained("/kaggle/working/llm-rag-finetuned-instruct-v1/best_model")

base_model.base_model.resize_token_embeddings(len(tokenizer))

model = PeftModel.from_pretrained(base_model, '/kaggle/working/llm-rag-finetuned-instruct-v1/best_model')
model = model.merge_and_unload()
model.eval()

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

`low_cpu_mem_usage` was None, now set to True since model is quantized.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

OSError: Incorrect path_or_model_id: '/kaggle/working/llm-rag-finetuned-instruct-v1/best_model'. Please provide either the path to a local folder or the repo_id of a model on the Hub.

In [ ]:
tokenizer.padding_side = "left"
model.config.pad_token_id = tokenizer.pad_token_id

In [ ]:
def left_pad(batch, pad_value):
  max_len = max(len(x) for x in batch)
  return torch.stack([
      torch.cat((
          torch.full((max_len - len(x), ), pad_value, dtype=x.dtype),
          x
      ))
      for x in batch
  ])

def collate_fn(batch):
  answers = []
  input_ids_list = []
  attention_mask_list = []

  for item in batch:
      labels = torch.tensor(item['labels'])
      input_ids = torch.tensor(item['input_ids'])
      attention_mask = torch.tensor(item['attention_mask'])

      answer_start = (
          (labels != -100)
          .nonzero(as_tuple=True)[0][0]
          .item()
      )

      prompt_ids = input_ids[:answer_start]
      prompt_mask = attention_mask[:answer_start]

      answers.append(
          tokenizer.decode(
              input_ids[answer_start:],
              skip_special_tokens=True
          )
      )

      input_ids_list.append(prompt_ids)
      attention_mask_list.append(prompt_mask)

  input_ids_list = left_pad(input_ids_list, tokenizer.pad_token_id)
  attention_mask_list = left_pad(attention_mask_list, 0)

  return {
      'input_ids': input_ids_list,
      'attention_mask': attention_mask_list,
      'answers': answers
  }

In [ ]:
def model_inference(dataloader):
    q_a_pairs = []
    bleu_score_list = []
    rouge_score_list = []
    bertscore_score_list = []

    model.eval()

    with torch.no_grad():
      for batch in dataloader:
        input_ids = batch['input_ids'].to(model.device)
        attention_mask = batch['attention_mask'].to(model.device)

        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=256,
            min_new_tokens=5,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            do_sample=False,
            repetition_penalty=1.1,
            stop_strings=["###", "\n\n-", "\n- Điều"],
            tokenizer = tokenizer
        )

        generated_tokens = outputs[:, input_ids.shape[1]:]

        pred_answers = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
        prompts = tokenizer.batch_decode(input_ids, skip_special_tokens=True)

        for idx, (predicted_answer, ground_truth, prompt_text) in enumerate(zip(pred_answers, batch['answers'], prompts)):
          metrics = compute_metrics(predicted_answer, ground_truth)

          q_a_pairs.append({
              'prompt': prompt_text,
              'pred_answer': predicted_answer,
              'ground_truth': ground_truth,
              'metrics': metrics
          })

          bleu_score_list.append(metrics['bleu'])
          rouge_score_list.append(metrics['rougeL'])
          bertscore_score_list.append(metrics['bertscore'])

    return q_a_pairs, bleu_score_list, rouge_score_list, bertscore_score_list

In [ ]:
import json
import matplotlib.pyplot as plt

def draw_metrics(metrics, values, file_name):
    plt.bar(metrics, values)
    plt.title('Mean Metrics')
    plt.xlabel('Metrics')
    plt.ylabel('Values')
    plt.savefig(f'figs/{file_name}', bbox_inches='tight', dpi=300)
    plt.show()


def write_inference(q_a_pairs, file_name):
    with open(f'/kaggle/working/outputs/{file_name}', 'w') as f:
      json.dump(q_a_pairs, f, ensure_ascii=False, indent=4)

In [ ]:
dev_dataloader = DataLoader(
    dev_dataset,
    batch_size=2,
    collate_fn=collate_fn,
    shuffle=False
)

q_a_pairs, bleu_score_list, rouge_score_list, bertscore_score_list = model_inference(dev_dataloader)

metrics = ['BLEU', 'ROUGEL', 'BERTSCORE']
values = [np.mean(bleu_score_list), np.mean(rouge_score_list), np.mean(bertscore_score_list)]

draw_metrics(metrics, values, '512tokens_3epoches_lora8+16_quantized_min_5_stopstring_betterdata')
write_inference(q_a_pairs, '512tokens_3epoches_lora8+16_quantized_min_5_stopstring_betterdata')

## Test

In [ ]:
test_dataset = test_dataset.map(preprocess)
test_dataset = test_dataset.map(tokenize, remove_columns=train_dataset.column_names)

In [ ]:
dev_dataloader = DataLoader(
    test_dataset,
    batch_size=2,
    collate_fn=collate_fn,
    shuffle=False
)

q_a_pairs, bleu_score_list, rouge_score_list, bertscore_score_list = model_inference(dev_dataloader)

metrics = ['BLEU', 'ROUGEL', 'BERTSCORE']
values = [np.mean(bleu_score_list), np.mean(rouge_score_list), np.mean(bertscore_score_list)]

draw_metrics(metrics, values, '768tokens_5epoches_lora8+16_quantized_min_5_stopstring')
write_inference(q_a_pairs, '768tokens_5epoches_lora8+16_quantized_min_5_stopstring')